# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GODRATAN/FlyRank_ML_Assignment/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Baseline rule

Prioritize pages that have meaningful search exposure but weaker-than-expected CTR for their observed search position. The rule should focus editorial attention on pages where improving the result snippet or content relevance may have a useful opportunity because the page is already receiving search visibility.

Reason codes:
- `HIGH_EXPOSURE_LOW_CTR` — meaningful impressions with weak CTR.
- `NORMAL_OPPORTUNITY` — lower-priority pages that do not meet the main threshold.

Action labels:
- `REVIEW_CTR` — review the page for a possible CTR/content improvement.
- `MONITOR` — keep the page under observation rather than prioritizing it immediately.

In [1]:
from google.colab import userdata
from huggingface_hub import login
from datasets import load_dataset

HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN)

march = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    data_files={
        "train": "hf://datasets/FlyRank/internship-warehouse/"
                 "fact_content_daily_performance/month=2026-03/data_0.parquet"
    },
    token=HF_TOKEN
)

df = march["train"].to_pandas()

print("March 2026 rows:", len(df))
print("Date range:", df["report_date"].min(), "to", df["report_date"].max())

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

March 2026 rows: 9841378
Date range: 2026-03-01 to 2026-03-31


In [3]:
import pandas as pd

# Signal 1 — Search exposure (impressions)

df["impression_bucket"] = pd.cut(
    df["gsc_impressions"],
    bins=[-1, 0, 10, 100, 500, float("inf")],
    labels=["0", "1-10", "11-100", "101-500", "500+"]
)

impression_check = (
    df["impression_bucket"]
    .value_counts(sort=False)
    .rename_axis("impression_bucket")
    .reset_index(name="n")
)

print(impression_check)

  impression_bucket        n
0                 0  6230317
1              1-10  1531634
2            11-100  1445944
3           101-500   532347
4              500+   101136


### Signal 1 verdict — MIXED

Impressions show that most rows have limited or no observed search exposure, while a smaller group has substantial exposure. This makes impressions useful as an opportunity/visibility signal, but the bucket distribution alone does not show that higher-impression pages are more likely to need a refresh.

Therefore, the signal is treated as directional rather than definitive.

In [4]:
# Signal 2 — CTR relative to average position

# Calculate CTR only where impressions are positive.
df["ctr"] = df["gsc_clicks"] / df["gsc_impressions"].replace(0, pd.NA)

df["position_bucket"] = pd.cut(
    df["gsc_avg_position"],
    bins=[-0.01, 3, 10, 20, 50, float("inf")],
    labels=["1-3", "4-10", "11-20", "21-50", "50+"]
)

ctr_check = (
    df.groupby("position_bucket", observed=False)
      .agg(
          n=("ctr", "count"),
          mean_ctr=("ctr", "mean")
      )
      .reset_index()
)

print(ctr_check)

  position_bucket        n  mean_ctr
0             1-3   727362  0.004756
1            4-10  1456122  0.003473
2           11-20   519223   0.00277
3           21-50   631491  0.001638
4             50+   276863  0.000494


### Signal 2 verdict — CONFIRMED

CTR declines as average search position worsens in the March 2026 slice. The measured mean CTR is highest for positions 1–3 (0.004756) and lowest for positions 50+ (0.000494). This supports using CTR relative to observed position as a directional signal for identifying pages that may deserve review.

This is an observed relationship, not a causal claim.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [5]:
# ML-07 — Build the baseline ranked queue

# Pages with enough search exposure and CTR below the typical level
# for their position bucket receive the main review action.

# Calculate the reference CTR for each position bucket.
position_ctr = (
    df.groupby("position_bucket", observed=False)["ctr"]
      .transform("mean")
)

# Main conditions
visible = df["gsc_impressions"] >= 100
low_ctr = df["ctr"] < position_ctr

# Transparent baseline score
df["baseline_score"] = (
    visible.astype(int) * low_ctr.astype(int) * df["gsc_impressions"]
)

# One reason code
df["reason_code"] = "NORMAL_OPPORTUNITY"
df.loc[
    (visible & low_ctr),
    "reason_code"
] = "HIGH_EXPOSURE_LOW_CTR"

# One action label
df["action"] = "MONITOR"
df.loc[
    (visible & low_ctr),
    "action"
] = "REVIEW_CTR"

# Rank highest-priority pages first
queue = df.sort_values(
    "baseline_score",
    ascending=False
).copy()

# Save the ranked queue
output_path = "work/outputs/baseline_action_score.csv"

import os
os.makedirs("work/outputs", exist_ok=True)

queue[
    [
        "report_date",
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ctr",
        "baseline_score",
        "reason_code",
        "action",
    ]
].to_csv(output_path, index=False)

print("Queue rows:", len(queue))
print("Rows selected for REVIEW_CTR:", (queue["action"] == "REVIEW_CTR").sum())
print("CSV written to:", output_path)

Queue rows: 9841378
Rows selected for REVIEW_CTR: 433716
CSV written to: work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.